In [1]:
import motsmeles
import numpy.random as rd
import numpy as np
import pandas as pd

## Définitions des fonctions qui seront utiles.

In [6]:
#Fonction qui répond à l'algorithme s'il peut ou non aller la où il souhaite aller.

def can(game,c):
    g = (c[1]-1) >= 0           # gauche
    d = (c[1]+1) < len(game.grid[0])  # droite
    h = (c[0]-1) >= 0           # haut
    b = (c[0]+1) < len(game.grid[:,0])  # bas
    
    # Toutes les directions (ordre : G, D, H, B, HG, HD, BG, BD)
    cap = [
        int(g),
        int(d),
        int(h),
        int(b),
        int(h and g),
        int(h and d),
        int(b and g),
        int(b and d)
    ]
    return [bool(x) for x in cap]

In [7]:
import numpy as np

# Fonctions directionnelles
def l(p):  return p - np.array([0, 1])   # gauche
def r(p):  return p + np.array([0, 1])   # droite
def u(p):  return p - np.array([1, 0])   # haut
def d(p):  return p + np.array([1, 0])   # bas
def ul(p): return p - np.array([1, 1])   # haut-gauche
def ur(p): return p - np.array([1, -1])  # haut-droite
def dl(p): return p + np.array([1, -1])  # bas-gauche
def dr(p): return p + np.array([1, 1])   # bas-droite

# --- Sélecteur de direction (indexé à partir de 0) ---
def m(n):
    """
    Renvoie la fonction de mouvement correspondant à n (0–7).
    0: gauche
    1: droite
    2: haut
    3: bas
    4: haut-gauche
    5: haut-droite
    6: bas-gauche
    7: bas-droite
    """
    moves = [l, r, u, d, ul, ur, dl, dr]
    if 0 <= n < len(moves):
        return moves[n]
    else:
        return None


In [8]:
#Fonction qui aide l'algorithme à savor si cette chose est réellement ce qu'il pense que ce soit. 
def isthat(game,p,v):
    return game.grid[tuple(p)] == v

In [9]:
#Permet de décomposer le mot qu'il souhaite chercher

def decword (game,word):
    return  list(word)
    

In [10]:
#Dis à l'algorithme où se trouve la lettre du mot qu'il cherche.

def where(game,word,l=0):
    w= decword(game,word)
    return np.argwhere(game.grid == w[l])

## Commençons par créer une neuronne qui permet à l'algorithme de trouver où se trouve un mot.

In [11]:
def solvew(game, w=0):
    word = game.words[w]
    first_letter = decword(game, word)[0]
    pos = where(game, first_letter, l=0)  # positions de la première lettre
    M = []

    for p in pos:              # pour chaque point de départ possible
        for k in range(8):     # pour chaque direction
            P = np.array(p)
            found = [tuple(P)]  # positions trouvées pour cette tentative
            success = True

            for i in range(1, len(word)):
                if not can(game,P)[k]:   # si on ne peut plus avancer dans cette direction
                    success = False
                    break
                P = m(k)(P)         # avance d’une case
                if not isthat(game, P, decword(game, word)[i]):  # mauvaise lettre
                    success = False
                    break
                found.append(tuple(P))

            if success:             # si le mot complet est trouvé
                M.extend(found)
                return M            # tu peux même return ici si tu veux arrêter dès qu’un mot est trouvé

    return []   # aucun mot trouvé

## Puis une neuronne qui lui permet de résoudre tous les mots.

In [12]:
import pandas as pd

def solve(game):
    sol = []

    # Récupérer toutes les solutions
    for k in range(len(game.words)):
        sol.append(solvew(game, w=k))  # solvew renvoie déjà une liste de positions

    # Convertir la liste de listes en DataFrame
    df = pd.DataFrame(sol, index=game.words)

    # Renommer les colonnes de manière explicite
    df.columns = [f"Pos{i}" for i in range(df.shape[1])]
    return df

## Pour tester l'algorithme.

In [989]:
game,answers=motsmeles.generate(
    [
        "PYTHON",
        "JACA",
        "CPP",
        "HTML",
        "CSS",
        "PHP",
        "RUBY",
        "SWIFT",
        "PERL",
        "RUST",
        "MAISON"
    ],
    width = 10, 
    height = 10)


In [990]:
solve(game)

,Pos0,Pos1,Pos2,Pos3,Pos4,Pos5
PYTHON,"(5, 2)","(5, 3)","(5, 4)","(5, 5)","(5, 6)","(5, 7)"
JACA,"(4, 9)","(3, 9)","(2, 9)","(1, 9)",None,None
CPP,"(0, 4)","(0, 5)","(0, 6)",None,None,None
HTML,"(5, 5)","(6, 4)","(7, 3)","(8, 2)",None,None
CSS,"(7, 6)","(7, 5)","(7, 4)",None,None,None
PHP,"(7, 2)","(8, 1)","(9, 0)",None,None,None
RUBY,"(5, 9)","(4, 8)","(3, 7)","(2, 6)",None,None
SWIFT,"(5, 0)","(4, 1)","(3, 2)","(2, 3)","(1, 4)",None
PERL,"(0, 7)","(1, 6)","(2, 5)","(3, 4)",None,None
RUST,"(0, 1)","(1, 1)","(2, 1)","(3, 1)",None,None


In [991]:
game.grid

array([['K', 'R', 'C', 'P', 'C', 'P', 'P', 'P', 'C', 'P'],
       ['O', 'U', 'A', 'Q', 'T', 'G', 'E', 'N', 'F', 'A'],
       ['D', 'S', 'R', 'F', 'G', 'R', 'Y', 'R', 'S', 'C'],
       ['B', 'T', 'I', 'V', 'L', 'I', 'G', 'B', 'X', 'A'],
       ['I', 'W', 'U', 'K', 'F', 'I', 'O', 'G', 'U', 'J'],
       ['S', 'D', 'P', 'Y', 'T', 'H', 'O', 'N', 'J', 'R'],
       ['E', 'H', 'N', 'T', 'T', 'G', 'P', 'J', 'U', 'F'],
       ['W', 'S', 'P', 'M', 'S', 'S', 'C', 'I', 'N', 'Q'],
       ['S', 'H', 'L', 'F', 'Q', 'A', 'M', 'X', 'X', 'F'],
       ['P', 'N', 'N', 'O', 'S', 'I', 'A', 'M', 'N', 'F']], dtype='<U1')

In [2]:
game,answers=motsmeles.generate(
    [
      "CHAUSSURE",
        "VOITURE"
    ],
    width = 12, 
    height = 12)


In [4]:
pd.DataFrame(game.grid)

,0,1,2,3,4,5,6,7,8,9,10,11
0,X,D,Y,M,G,P,H,T,B,F,F,H
1,V,S,Z,E,J,R,V,K,K,T,D,G
2,O,H,T,T,B,O,E,H,M,W,K,C
3,V,O,P,Q,I,E,K,C,Y,G,H,K
4,W,O,Y,T,H,J,W,G,H,A,P,C
5,F,B,U,U,K,E,B,B,U,J,F,G
6,W,R,X,E,A,P,Y,S,W,K,T,V
7,E,H,O,U,D,R,S,K,S,L,A,Q
8,J,D,E,M,N,U,F,V,J,A,Z,R
9,L,F,S,H,R,E,L,J,E,G,P,B


In [5]:
solve(game)

NameError: name 'solve' is not defined